# Training & Inference on Kaggle
Pipeline for prepatched 224x224 tiles stored under `/kaggle/input/<dataset>`. Uses repo defaults (lazy_loaders.DEFAULT_LABEL_MAP: Luminal A=0, Luminal B=1, HER2(+) =2, Triple negative=3).


In [ ]:
from pathlib import Path
import os, sys, random, subprocess, json
import numpy as np
import torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
REPO_DIR = Path("/kaggle/working/an2dl-challenges-25-26")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/asarraa/an2dl-challenges-25-26", str(REPO_DIR)], check=True)

WORK_DIR = REPO_DIR / "challenge2"
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

print(f"Working dir: {WORK_DIR}")

!pip install -q -r "{REPO_DIR / 'requirements.txt'}" comet_ml torchsummary


## Paths & label mapping


In [ ]:
import pandas as pd
import lazy_loaders
import config

# If you renamed the dataset on Kaggle, set this to Path("/kaggle/input/<your-dataset>")
USER_DATASET_DIR = None

PREPROCESS_NAME = "preprocess_v1"
PREFERRED_NAMES = [PREPROCESS_NAME, PREPROCESS_NAME.replace("_", "-")]
INPUT_ROOT = Path("/kaggle/input")

def is_valid_preprocessed(path: Path) -> bool:
    return ((path / "train_patches.csv").exists() and (path / "test_patches.csv").exists()
            and (path / "train" / "images").exists() and (path / "test" / "images").exists())

candidates = []
if USER_DATASET_DIR is not None:
    candidates.append(Path(USER_DATASET_DIR))
candidates.extend([INPUT_ROOT / name for name in PREFERRED_NAMES])
if INPUT_ROOT.exists():
    for entry in INPUT_ROOT.iterdir():
        if entry.is_dir():
            candidates.append(entry)
candidates.append(WORK_DIR / "data" / "preprocessed" / PREPROCESS_NAME)

PREPROCESSED_DIR = None
for path in candidates:
    if is_valid_preprocessed(path):
        PREPROCESSED_DIR = path
        break

if PREPROCESSED_DIR is None:
    raise FileNotFoundError("Preprocessed dataset not found under /kaggle/input. Set USER_DATASET_DIR or update PREPROCESS_NAME.")

TRAIN_CSV = PREPROCESSED_DIR / "train_patches.csv"
TEST_CSV = PREPROCESSED_DIR / "test_patches.csv"

LABEL_MAP = lazy_loaders.DEFAULT_LABEL_MAP
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

mask_train_dir = PREPROCESSED_DIR / "train" / "masks"
mask_test_dir = PREPROCESSED_DIR / "test" / "masks"
ADD_MASK_CHANNEL = mask_train_dir.exists() and mask_test_dir.exists()

# Update config so any utility using it points to the patched data
config.TRAIN_DIR = PREPROCESSED_DIR / "train"
config.TEST_DIR = PREPROCESSED_DIR / "test"
config.LABELS_CSV = TRAIN_CSV

print(f"Using preprocessed data at: {PREPROCESSED_DIR}")
print(f"Label mapping: {LABEL_MAP}")
print(f"Masks detected: {ADD_MASK_CHANNEL}")


## Automated augmentation (RandAugment + geo/color)


In [ ]:
import torchvision.transforms.v2 as transforms

# RandAugment explores perturbations automatically; we keep light geo/color jitter for stability
TRAIN_AUG = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandAugment(num_ops=2, magnitude=9),
])


## Dataloaders


In [ ]:
import os

BATCH_SIZE = int(os.environ.get("BATCH_SIZE", config.LOADER_PARAMS["batch_size"]))
# If masks are present (4 channels) lower batch size a bit to avoid OOM on Kaggle GPUs
if ADD_MASK_CHANNEL:
    BATCH_SIZE = max(1, BATCH_SIZE // 2)
config.LOADER_PARAMS["batch_size"] = BATCH_SIZE

train_loader, val_loader, input_shape = lazy_loaders.get_loaders(
    augmentation=TRAIN_AUG,
    batch_size=BATCH_SIZE,
    base_path=PREPROCESSED_DIR,
    add_mask_channel=ADD_MASK_CHANNEL,
    return_label_map=True,
)

test_loader, _ = lazy_loaders.get_test_loader(
    batch_size=BATCH_SIZE,
    base_path=PREPROCESSED_DIR,
    add_mask_channel=ADD_MASK_CHANNEL,
)

print(f"Using batch size: {BATCH_SIZE} (mask channel: {ADD_MASK_CHANNEL})")
print(f"Input shape: {input_shape}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## Training (ResNet backbone on RGB patches)


In [ ]:
from launch_training import start_training

TRAINING_PARAMS = config.TRAINING_DEFAULTS.copy()
TRAINING_PARAMS.update({
    "epochs": 40,
    "learning_rate": 3e-4,
    "patience": 8,
    "verbose": 1,
    "l1_lambda": 0,
    "l2_lambda": 0,
})

MODEL_PARAMS = config.RESNET_DEFAULTS.copy()
MODEL_PARAMS.update({
    "input_shape": input_shape,
    "num_classes": len(label_map),
    "backbone": "resnet18",
    "use_pretrained": True,
    "input_channels": input_shape[0],  # 3 for RGB tiles, 4 if masks are present
})

trained_model, history = start_training(
    model_name="HistologyResNet",
    model_params=MODEL_PARAMS,
    training_params=TRAINING_PARAMS,
    device=device,
    train_loader=train_loader,
    val_loader=val_loader,
    data_input_shape=input_shape,
)

print("Training done. Last val F1:", history['val_f1'][-1])


## Pick the latest checkpoint


In [ ]:
from datetime import datetime

def latest_checkpoint(registry_path):
    with open(registry_path) as f:
        data = json.load(f)
    if not data:
        raise RuntimeError("Registry is empty. Run training first or place an existing registry.json.")
    latest_id, latest_entry = max(data.items(), key=lambda kv: kv[1].get("timestamp", ""))
    return latest_id, Path(latest_entry["model_path"])

REGISTRY_PATH = WORK_DIR / "experiments" / "registry.json"
RUN_ID, MODEL_WEIGHTS_PATH = latest_checkpoint(REGISTRY_PATH)
print(f"Latest run: {RUN_ID}")
print(f"Checkpoint: {MODEL_WEIGHTS_PATH}")


## Inference + majority voting


In [ ]:
from tqdm.auto import tqdm
import models


def load_model_for_inference(checkpoint_path):
    model = models.HistologyResNet(
        num_classes=len(label_map),
        use_pretrained=False,
        backbone="resnet18",
        input_channels=input_shape[0],
    )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


model_for_eval = load_model_for_inference(MODEL_WEIGHTS_PATH)

all_tile_names, all_preds = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader):
        imgs = imgs.to(device)
        logits = model_for_eval(imgs)
        preds = logits.argmax(dim=1).cpu().tolist()
        all_tile_names.extend(names)
        all_preds.extend(preds)

df_tiles = pd.DataFrame({"sample_index": all_tile_names, "pred_idx": all_preds})
df_meta = pd.read_csv(TEST_CSV)

merged = df_tiles.merge(df_meta[["sample_index", "original_sample"]], on="sample_index", how="left")
final_rows = []
for slide, group in merged.groupby("original_sample"):
    best_idx = group["pred_idx"].value_counts().idxmax()
    final_rows.append({
        "sample_index": slide,
        "label": inv_label_map[best_idx],
        "label_id": int(best_idx),
    })

submission = pd.DataFrame(final_rows).sort_values("sample_index")
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)

print(submission.head())
print(f"Submission saved to: {SUBMISSION_PATH}")
